In [ ]:
"""
NO ADDITIONAL RESULTS.. USELESS SCRIPT



Search for LPG points of sale (filling + reseller) in all capitals and second cities
of continental sub-Saharan Africa (including Madagascar).
Sources: OpenStreetMap (Overpass) + Google Places (multilingual).
Output: single CSV, JSON, GeoPackage for all cities, plus a summary CSV with statistics.

Keywords are commented out for cost control.
eta: 45 minutes with both sources enabled
"""

import requests
import time
import csv
import json
import os
import math
import random
from datetime import datetime
from collections import Counter

# ======================== CONFIGURATION ========================
USE_OSM = True                     # ## CHANGE ## flag to enable/disable OSM 
USE_GOOGLE = False               # pay attention to Google API costs! flag to enable/disable Google Places
API_KEY = "MM"  # <-- insert valid key
ENABLE_RESUME = False                 # If True, skip cities already processed in a previous run
PROCESSED_CITIES_FILE = "processed_cities.txt"   # file inside OUTPUT_ROOT that stores city names



RADIUS = 50000                     # radius in metres for OSM & Google Nearby Search

# Output root folder
OUTPUT_ROOT = "output_equatorial_guinea_BigCity"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Timestamp for final output files
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Global list that holds all points from all cities (used for intermediate saves)
global_all_results = []


# ======================== KEYWORDS ========================
# ----- RESELLER (reseller) -----
RESELLER_KEYWORDS = {
    'en': [
        "LPG refill station", "gas station with LPG", "LPG filling point",
        "butane gas", "LPG dealer", "gas cylinder exchange",
        "cooking gas", "LPG"
    ],
    'fr': [
        "station de remplissage de GPL", "station-service avec GPL",
        "point de remplissage de GPL",
        "revendeur de GPL", "échange de bouteilles de gaz",
        "gaz de cuisine", "GPL", "gaz butane"
    ],
    'pt': [
        "estação de recarga de GLP", "posto de gasolina com GLP",
        "gás butano", "ponto de enchimento de GLP",
        "revendedor de GLP", "troca de botijão de gás",
        "gás de cozinha", "GLP"
    ],
    'sw': [
        "kituo cha kujaza LPG", "kituo cha mafuta chenye LPG",
        "sehemu ya kujaza LPG", "muuzaji wa LPG",
        "kubadilisha mitungi ya gesi", "gesi ya butane",
        "gesi ya kupikia", "LPG"
    ],
    'ar': [
         "محطة تعبئة غاز البترول المسال", "محطة وقود مع غاز البترول المسال",
         "غاز البيوتان", "نقطة تعبئة غاز البترول المسال",
         "تاجر غاز البترول المسال", "استبدال اسطوانات الغاز",
        "غاز الطبخ", "غاز البترول المسال"
    ],
    'es': [
        "estación de recarga de GLP",
        "gasolinera con GLP",
        "punto de llenado de GLP",
        "gas butano",
        "distribuidor de GLP",
        "intercambio de bombonas de gas",
        "gas de cocina",
        "GLP"
    ],
        

}

# ----- FILLING PLANTS (filling) -----
FILLING_KEYWORDS = {
    'en': [
        "LPG cylinder filling plant", "LPG bottling plant",
         "gas bottling plant", "LPG filling plant",
        "cylinder filling station", "LPG bottling station"
    ],
    'fr': [
        "usine de remplissage de bouteilles de GPL", "usine d'embouteillage de GPL",
         "usine d'embouteillage de gaz", "usine de remplissage de GPL",
        "station de remplissage de bouteilles", "station d'embouteillage de GPL"
    ],
    'pt': [
        "fábrica de enchimento de botijas de GLP", "fábrica de engarrafamento de GLP",
         "fábrica de engarrafamento de gás", "fábrica de enchimento de GLP",
        "estação de enchimento de botijas", "estação de engarrafamento de GLP"
    ],
    'sw': [
        "kiwanda cha kujaza mitungi ya LPG",
        "kiwanda cha kujaza LPG",
         "kiwanda cha kujaza gesi", "kiwanda cha kusindika LPG",
        "kituo cha kujaza mitungi", "kituo cha kujaza LPG"
    ],

    'es': [
        "planta envasadora de GLP",
        "planta embotelladora de GLP",
        "planta de envasado de gas",
        "planta de llenado de GLP",
        "estación de llenado de cilindros",
        "estación de envasado de GLP"
    ],


    'ar': [
        "مصنع تعبئة اسطوانات غاز البترول المسال", "مصنع تعبئة غاز البترول المسال",
         "مصنع تعبئة الغاز", "محطة تعبئة اسطوانات الغاز",
        "محطة تعبئة غاز البترول المسال", "محطة تعبئة الغاز"
    ]
}

# ======================== LANGUAGE MAP PER COUNTRY ========================
COUNTRY_LANGUAGES = {
    'angola':             ['en', 'pt'],
    'benin':              ['en', 'fr'],
    'botswana':           ['en'],
    'burkina_faso':       ['en', 'fr'],
    'burundi':            ['en', 'fr', 'sw'],
    'cameroon':           ['en', 'fr'],
    'central_african_republic': ['en', 'fr'],
    'chad':               ['en', 'fr', 'ar'],
    'congo':              ['en', 'fr'],
    'congo_drc':          ['en', 'fr', 'sw'],
    'cote_divoire':       ['en', 'fr'],
    'djibouti':           ['en', 'fr', 'ar'],
    'equatorial_guinea':  ['en', 'pt', 'fr', 'es'],
    'eritrea':            ['en', 'ar'],
    'eswatini':           ['en'],
    'ethiopia':           ['en'],
    'gabon':              ['en', 'fr'],
    'gambia':             ['en'],
    'ghana':              ['en'],
    'guinea':             ['en', 'fr'],
    'guinea_bissau':      ['en', 'pt'],
    'kenya':              ['en', 'sw'],
    'lesotho':            ['en'],
    'liberia':            ['en'],
    'madagascar':         ['en', 'fr'],
    'malawi':             ['en'],
    'mali':               ['en', 'fr'],
    'mauritania':         ['en', 'fr', 'ar'],
    'mozambique':         ['en', 'pt'],
    'namibia':            ['en'],
    'niger':              ['en', 'fr'],
    'nigeria':            ['en'],
    'rwanda':             ['en', 'fr', 'sw'],
    'senegal':            ['en', 'fr'],
    'sierra_leone':       ['en'],
    'somalia':            ['en', 'ar'],
    'south_africa':       ['en'],
    'south_sudan':        ['en', 'ar', 'sw'],
    'sudan':              ['en', 'ar'],
    'tanzania':           ['en', 'sw'],
    'togo':               ['en', 'fr'],
    'uganda':             ['en', 'sw'],
    'zambia':             ['en'],
    'zimbabwe':           ['en']
}

# ======================== CITY LIST ========================
# Central coordinates for capital and second city; the script performs a single 50 km circle search.
CITIES_TO_PROCESS = [
    
    # Equatorial Guinea
    {'city': 'malabo',      'country': 'equatorial_guinea','lat': 3.750, 'lon': 8.783},
    {'city': 'bata',        'country': 'equatorial_guinea','lat': 1.867, 'lon': 9.767}
    
]

# ======================== OSM SEARCH FUNCTION (single circle) ========================
def osm_search(lat, lon, radius=50000, max_retries=3):
    """
    Performs a single Overpass API query for a circular area around (lat, lon).
    Returns a list of place dictionaries.
    """
    delta = radius / 111000.0
    min_lat = lat - delta
    max_lat = lat + delta
    min_lon = lon - delta
    max_lon = lon + delta

    # Simplified, non‑redundant query
    query = f"""
    [out:json];
    (
        node["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["amenity"="fuel"]["fuel:lpg"~"yes|only|lpg|LPG|GPL|GLP"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["shop"="gas_cooking"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        node["man_made"="storage_tank"]["content"~"lpg|gpl|glp|gas|butane|propane"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["industrial"="gas"]({min_lat},{min_lon},{max_lat},{max_lon});
        way["man_made"="storage_tank"]["content"~"lpg|gpl|glp|gas|butane|propane"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    url = "https://overpass-api.de/api/interpreter"
    headers = {
        'User-Agent': 'LPG-Research-Script/1.0 (contact: your-email@example.com)',
        'Accept': 'application/json'
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, data={'data': query}, headers=headers, timeout=60)
            if response.status_code == 200:
                data = response.json()
                break
            elif response.status_code in [429, 504]:
                wait = (2 ** attempt) + random.uniform(0, 1)   # exponential backoff: 1, 2, 4 seconds
                print(f"  ⚠️ OSM {response.status_code} - retry in {wait:.1f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            else:
                return [], response.status_code
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            else:
                return [], f"Exception({e})"
    else:
        # loop finished without success
        return [], response.status_code

    results = []
    for elem in data.get('elements', []):
        if elem['type'] == 'node':
            lat_place = elem.get('lat')
            lon_place = elem.get('lon')
        else:
            center = elem.get('center', {})
            lat_place = center.get('lat')
            lon_place = center.get('lon')
        if lat_place is None or lon_place is None:
            continue
        tags = elem.get('tags', {})
        if 'amenity' in tags and tags['amenity'] == 'fuel':
            ptype = 'fuel_station'
        elif 'shop' in tags:
            ptype = tags['shop']
        else:
            ptype = 'lpg_related'

        # Determine category
        is_fuel_station = tags.get('amenity') == 'fuel' and (
            tags.get('fuel:lpg') or tags.get('fuel:GPL') or tags.get('fuel:lpg') in ['yes', 'only']
        )
        is_storage_tank = tags.get('man_made') == 'storage_tank' and tags.get('content', '').lower() in ['lpg', 'gpl', 'glp', 'gas', 'butane', 'propane']
        is_industrial_gas = tags.get('industrial') == 'gas'
        category = 'filling' if (is_fuel_station or is_storage_tank or is_industrial_gas) else 'reseller'

        name = tags.get('name', '')
        results.append({
            'place_id': f"osm_{elem['id']}",
            'name': name,
            'address': '',
            'lat': lat_place,
            'lng': lon_place,
            'types': ptype,
            'keyword': 'osm',
            'search_lat': lat,
            'search_lon': lon,
            'rating': None,
            'user_ratings_total': 0,
            'source': 'osm',
            'language': '',
            'category': category
        })
    return results, None

# ======================== GOOGLE PLACES FUNCTIONS ========================
def google_nearby_search(lat, lon, keyword, lang, category):
    url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    results = []
    params = {
        'location': f"{lat},{lon}",
        'radius': RADIUS,
        'keyword': keyword,
        'language': lang,
        'key': API_KEY
    }
    while True:
        resp = requests.get(url, params=params)
        data = resp.json()
        if data['status'] not in ['OK', 'ZERO_RESULTS']:
            error_msg = data.get('error_message', '')
            print(f"  ⚠️ Google status {data['status']} for {keyword}_{lang}: {error_msg}")
            break
        for place in data.get('results', []):
            results.append({
                'place_id': place['place_id'],
                'name': place.get('name', ''),
                'address': place.get('vicinity', ''),
                'lat': place['geometry']['location']['lat'],
                'lng': place['geometry']['location']['lng'],
                'types': ', '.join(place.get('types', [])),
                'keyword': f"{keyword}_{lang}",
                'search_lat': lat,
                'search_lon': lon,
                'rating': place.get('rating'),
                'user_ratings_total': place.get('user_ratings_total', 0),
                'source': 'google',
                'language': lang,
                'category': category
            })
        if 'next_page_token' in data:
            params['pagetoken'] = data['next_page_token']
            time.sleep(2)
        else:
            break
    return results

# ======================== SAVE GLOBAL FILES (intermediate saves) ========================
def save_global_files(results, output_root, timestamp):
    """
    Saves (overwrites) the global CSV, JSON and GeoPackage files.
    This function is called after each city to provide a crash-safe backup.
    """
    if not results:
        return
    base = os.path.join(output_root, f"all_africa_combined_{timestamp}")
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total','source','language','category']

    # CSV
    with open(f"{base}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    # JSON
    with open(f"{base}.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    # GeoPackage (separate layers for filling and reseller)
    try:
        import geopandas as gpd
        import pandas as pd
        from shapely.geometry import Point
        keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                     'rating','user_ratings_total','source','language','category']
        filling = [r for r in results if r['category'] == 'filling']
        reseller = [r for r in results if r['category'] != 'filling']
        if filling:
            df = pd.DataFrame(filling)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg", driver='GPKG', layer='filling')
        if reseller:
            df = pd.DataFrame(reseller)
            geom = [Point(x,y) for x,y in zip(df['lng'], df['lat'])]
            gdf = gpd.GeoDataFrame(df[[c for c in keep_cols if c in df.columns]], geometry=geom, crs="EPSG:4326")
            gdf['lat'] = df['lat']; gdf['lon'] = df['lng']
            gdf.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg", driver='GPKG', layer='reseller')
    except ImportError:
        pass  # GeoPandas not installed

# ======================== PROCESS ONE CITY ========================
def process_city(city_config, country_name, country_langs):
    city_name = city_config['city']
    lat_c, lon_c = city_config['lat'], city_config['lon']

  
    city_results = []
    osm_errors = Counter()

    # ---------- OSM ----------
    # ... inside process_city, after calling osm_search(lat_c, lon_c, RADIUS)
    if USE_OSM:
        time.sleep(2)   # pre‑call pause to respect rate limits
        osm_places, err_code = osm_search(lat_c, lon_c, RADIUS, max_retries=5)   # use more retries first

        if not osm_places and err_code is not None:
            # Fallback: 2x2 grid around the city centre (original method)
            print(f"  ↳ Single circle failed (err {err_code}), switching to 4‑point grid.")
            
            # 4 punti: angoli di un quadrato di 0.2° intorno al centro
            offsets = [(-0.1, -0.1), (-0.1, 0.1), (0.1, -0.1), (0.1, 0.1)]
            for dlat, dlon in offsets:
                glat = lat_c + dlat
                glon = lon_c + dlon
                time.sleep(2)
                try:
                    sub_places, sub_err = osm_search(glat, glon, RADIUS, max_retries=5)
                    if sub_places:
                        osm_places.extend(sub_places)
                except Exception as e:
                    print(f"  ⚠️ Grid point ({glat},{glon}) failed: {e}")

            # Deduplicate
            seen = {}
            for p in osm_places:
                if p['place_id'] not in seen:
                    seen[p['place_id']] = p
            osm_places = list(seen.values())
            
            err_code = None if osm_places else err_code

        if not err_code:
            city_results.extend(osm_places)
        else:
            osm_errors[err_code] += 1

    # ---------- Google ----------
    if USE_GOOGLE and API_KEY != "LA_TUA_API_KEY_GOOGLE":
        # Filling
        for lang in country_langs:
            if lang not in FILLING_KEYWORDS: continue
            for kw in FILLING_KEYWORDS[lang]:
                try:
                    places = google_nearby_search(lat_c, lon_c, kw, lang, 'filling')
                    city_results.extend(places)
                except Exception as e:
                    print(f"  ⚠️ Google filling {kw}_{lang}: {e}")
                time.sleep(0.4)
        # Reseller
        for lang in country_langs:
            if lang not in RESELLER_KEYWORDS: continue
            for kw in RESELLER_KEYWORDS[lang]:
                try:
                    places = google_nearby_search(lat_c, lon_c, kw, lang, 'reseller')
                    city_results.extend(places)
                except Exception as e:
                    print(f"  ⚠️ Google reseller {kw}_{lang}: {e}")
                time.sleep(0.4)


    # Deduplica per place_id (keep the first)
    unique_places_dict = {}   # place_id → record
    for place in city_results:
        pid = place['place_id']
        if pid not in unique_places_dict:
            unique_places_dict[pid] = place
        else:
        # se la nuova occorrenza è filling e quella già salvata no, sostituisci
            if place['category'] == 'filling' and unique_places_dict[pid]['category'] != 'filling':
                unique_places_dict[pid] = place
    city_results = list(unique_places_dict.values())


    # Ricalcola solo sui risultati Google unici
    google_results = [r for r in city_results if r['source'] == 'google']
    lang_counter = Counter(r['language'] for r in google_results) if google_results else Counter()

    

    # Source breakdown
    osm_total = sum(1 for r in city_results if r['source'] == 'osm')
    google_total = len(city_results) - osm_total
    filling = [r for r in city_results if r['category'] == 'filling']
    reseller = [r for r in city_results if r['category'] == 'reseller']


    if osm_errors:
        print(f"  OSM HTTP errors: {dict(osm_errors)}")

            
    # One compact line per city
    print(f"[{city_name.upper()}] {len(city_results)} pts | OSM {osm_total} Google {google_total} | Filling {len(filling)} Reseller {len(reseller)}")


        

    return city_results, lang_counter

# ======================== MAIN ========================
def main():
    global global_all_results
    summary_rows = []   # will hold dicts for summary CSV
    processed_cities = set()



    # --- 1. RESUME LOGIC: Load list of already processed cities ---
    if ENABLE_RESUME:
        resume_path = os.path.join(OUTPUT_ROOT, PROCESSED_CITIES_FILE)
        if os.path.exists(resume_path):
            with open(resume_path, 'r', encoding='utf-8') as f:
                processed_cities = set(line.strip().lower() for line in f if line.strip())
            print(f"--- Resume enabled: {len(processed_cities)} cities already processed ---")
        else:
            print("--- Resume enabled but no processed_cities file found. Starting fresh. ---")
    # ----------------------------------------------------------------



    # Group cities by country
    country_cities = {}
    for cfg in CITIES_TO_PROCESS:
        country = cfg['country']
        country_cities.setdefault(country, []).append(cfg)

    for country, city_list in country_cities.items():

        langs = COUNTRY_LANGUAGES.get(country, ['en']) 
        country_results = []

        for cfg in city_list:


            city_name = cfg['city']

            # --- 2. SKIP CHECK: If the city is in the set, skip it ---
            if city_name in processed_cities:
                print(f"⏩ Skipping {city_name} (already in resume file)")
                continue

            #normal processing
            city_results, lang_cnt = process_city(cfg, country, langs)
            country_results.extend(city_results)
            # Add to global list and save intermediate
            global_all_results.extend(city_results)
            save_global_files(global_all_results, OUTPUT_ROOT, TIMESTAMP)   # crash-safe backup
            # Persist the processed city immediately
            with open(os.path.join(OUTPUT_ROOT, PROCESSED_CITIES_FILE), 'a', encoding='utf-8') as f:
                f.write(city_name + '\n')
            processed_cities.add(city_name)
            time.sleep(2)

            # Build summary row for this city
            osm_city = sum(1 for r in city_results if r['source'] == 'osm')
            google_city = len(city_results) - osm_city
            filling_city = sum(1 for r in city_results if r['category'] == 'filling')
            reseller_city = sum(1 for r in city_results if r['category'] == 'reseller')
            # Language breakdown as string
            lang_str = ", ".join(f"{l}:{cnt}({cnt*100//sum(lang_cnt.values())}%)" for l, cnt in lang_cnt.most_common()) if lang_cnt else ""
            # Cerca questo blocco in fondo a main()
            summary_rows.append({
                'City': cfg['city'].upper(),     
                'Country': country.upper(),       
                'Total_Points': len(city_results), 
                'Total_OSM': osm_city,
                'Total_Google': google_city,
                'Total_Filling': filling_city,
                'Total_Reseller': reseller_city,
                'Google_Languages_pct': lang_str   
            })

        # Country statistics
        osm = [r for r in country_results if r['source']=='osm']
        gg = [r for r in country_results if r['source']=='google']
        filling = [r for r in country_results if r['category']=='filling']
        reseller = [r for r in country_results if r['category']=='reseller']
        print(f"[{country.upper()}] {len(country_results)} pts total | OSM {len(osm)} Google {len(gg)} | Filling {len(filling)} Reseller {len(reseller)}")

    # Save final global files (already up‑to‑date from last city, but just in case)
    save_global_files(global_all_results, OUTPUT_ROOT, TIMESTAMP)
    print(f"\n\nGlobal dataset saved with prefix 'all_africa_combined_{TIMESTAMP}'")

    # Save summary CSV
    summary_path = os.path.join(OUTPUT_ROOT, f"summary_{TIMESTAMP}.csv")
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['City', 'Country', 'Total_Points', 'Total_OSM', 'Total_Google',
                      'Total_Filling', 'Total_Reseller', 'Google_Languages_pct']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Summary CSV saved: {summary_path}")

    print("Processing completed.")

if __name__ == "__main__":
    if USE_GOOGLE and API_KEY == "LA_TUA_API_KEY_GOOGLE":
        print("⚠️ Insert a valid Google API key in the API_KEY variable.")
    main() 



    #note console output doesnt include spanish for equatorial guinea, but output files were adjusted and include spanish keywords for that country.

[MALABO] 24 pts | OSM 21 Google 3 | Filling 22 Reseller 2
[BATA] 1 pts | OSM 0 Google 1 | Filling 1 Reseller 0
[EQUATORIAL_GUINEA] 25 pts total | OSM 21 Google 4 | Filling 23 Reseller 2


Global dataset saved with prefix 'all_africa_combined_20260429_153937'
Summary CSV saved: output_equatorial_guinea_BigCity\summary_20260429_153937.csv
Processing completed.
